# Bitcoin Market Dynamics & On-Chain Network Activity Baseline Research

This notebook provides reproducible exploratory analysis on the interaction between Bitcoin exchange market dynamics (Coinbase BTC-USD) and on-chain network fundamentals (Coin Metrics transaction throughput and active address participation).

### Scope & Research Principles
- **Conformed Analytical Mart**: Exclusively consumes `mart_btc_market_and_network_daily`.
- **Zero Embedded ETL**: Contains no HTTP clients, raw data parsing, or ingestion watermarks. All transformations occur upstream in the core engineering pipeline.
- **Reproducible & Decoupled**: Consumes data via the `bitcoin_data_platform.serving` query layer and DuckDB analytical views.

In [ ]:
from pathlib import Path

from bitcoin_data_platform.serving.query_service import QueryService
from bitcoin_data_platform.storage.duckdb_manager import DuckDBManager

## 1. Connect to DuckDB Research Serving Layer

Connect to the DuckDB analytical database or initialize an in-memory instance referencing curated data partitions.

In [ ]:
# Determine project paths
is_repo = (Path("..") / "pyproject.toml").exists()
repo_root = Path("..").resolve() if is_repo else Path(".").resolve()
db_path = repo_root / "data" / "state" / "platform.duckdb"
curated_dir = repo_root / "data" / "curated"

# Connect to existing database or in-memory view catalog
if db_path.exists():
    manager = DuckDBManager(db_path=db_path, curated_dir=curated_dir)
else:
    manager = DuckDBManager(db_path=":memory:", curated_dir=curated_dir)
    with manager:
        manager.initialize()

serving = QueryService(manager)
print("Research Serving Layer initialized successfully.")

## 2. Query Conformed Cross-Domain Mart

Query `mart_btc_market_and_network_daily` to extract daily trading metrics alongside on-chain network activity.

In [ ]:
query = """
SELECT
    trade_date_utc,
    asset,
    market_open_usd,
    market_high_usd,
    market_low_usd,
    market_close_usd,
    market_volume_btc,
    is_market_day_complete,
    transaction_count,
    active_addresses_count,
    tx_per_active_address,
    ROUND(CAST((market_close_usd - market_open_usd) /
          NULLIF(market_open_usd, 0) * 100 AS DOUBLE), 4) AS daily_return_pct,
    ROUND(CAST((market_high_usd - market_low_usd) /
          NULLIF(market_low_usd, 0) * 100 AS DOUBLE), 4) AS high_low_spread_pct
FROM mart_btc_market_and_network_daily
ORDER BY trade_date_utc ASC;
"""

table = serving.execute(query)
print(f"Loaded {table.num_rows} records. Schema columns:")
for col in table.column_names:
    print(f"  - {col}")

## 3. Summary & Distributional Statistics

Compute aggregate statistics across price action and network throughput.

In [ ]:
summary_sql = """
SELECT
    COUNT(*) AS total_days,
    ROUND(AVG(CAST(market_close_usd AS DOUBLE)), 2) AS avg_close_usd,
    ROUND(MIN(CAST(market_close_usd AS DOUBLE)), 2) AS min_close_usd,
    ROUND(MAX(CAST(market_close_usd AS DOUBLE)), 2) AS max_close_usd,
    ROUND(AVG(CAST(market_volume_btc AS DOUBLE)), 2) AS avg_daily_volume_btc,
    ROUND(AVG(CAST(transaction_count AS DOUBLE)), 0) AS avg_tx_count,
    ROUND(AVG(CAST(active_addresses_count AS DOUBLE)), 0) AS avg_active_addresses,
    ROUND(AVG(tx_per_active_address), 4) AS avg_tx_per_active_address
FROM mart_btc_market_and_network_daily;
"""

summary_table = serving.execute(summary_sql)
for row in summary_table.to_pylist():
    for k, v in row.items():
        print(f"{k:30s}: {v}")

## 4. Cross-Domain Correlation Analysis

Evaluate Pearson correlation between exchange trading volume and on-chain network metrics.

In [ ]:
correlation_sql = """
SELECT
    ROUND(CORR(CAST(market_volume_btc AS DOUBLE),
               CAST(transaction_count AS DOUBLE)), 4) AS corr_volume_tx_count,
    ROUND(CORR(CAST(market_volume_btc AS DOUBLE),
               CAST(active_addresses_count AS DOUBLE)), 4) AS corr_volume_active_addr,
    ROUND(CORR(CAST(market_close_usd AS DOUBLE),
               CAST(active_addresses_count AS DOUBLE)), 4) AS corr_price_active_addr,
    ROUND(CORR(tx_per_active_address,
               CAST(market_volume_btc AS DOUBLE)), 4) AS corr_velocity_volume
FROM mart_btc_market_and_network_daily
WHERE market_volume_btc IS NOT NULL AND transaction_count IS NOT NULL;
"""

corr_table = serving.execute(correlation_sql)
for row in corr_table.to_pylist():
    for k, v in row.items():
        print(f"{k:30s}: {v}")

## 5. Conclusion & Analytical Takeaways

- **Serving Layer Decoupling**: Downstream analytics are completely isolated from upstream ingestion and storage formats.
- **Zero Embedded ETL**: Notebook execution is fast, safe, and reproducible without side-effects or network dependencies.
- **Precision Retention**: Monomorphic Arrow Tables preserve exact decimal and timestamp precision for downstream mathematical modeling.